# EEG de repouso no Parkinson: execução completa

Executa a pipeline do download às figuras. Cada etapa roda apenas se estiver `True` em `ETAPAS`.
O download e o pré-processamento retomam de onde pararam se a execução for interrompida.

Antes de executar, instale as dependências (`pip install -r requirements.txt`) e confira os
parâmetros da análise pareada em `eeg_pd/config.py` (`REFERENCIA_PRINCIPAL` e `MARGEM_ERRO_DELTA`).

In [ ]:
import importlib
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

RAIZ_PROJETO = Path.cwd() if (Path.cwd() / 'eeg_pd').is_dir() else Path.cwd().parent
os.chdir(RAIZ_PROJETO)
sys.path.insert(0, str(RAIZ_PROJETO))
from eeg_pd import cli
from eeg_pd.config import Caminhos

RAIZ = RAIZ_PROJETO / 'dados'
RAIZ_EXTERNA = RAIZ_PROJETO / 'dados_externo'
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
N_JOBS_PREPROCESSAMENTO = min(N_JOBS, 3)   # ~1-1,5 GB de memória por processo
N_PERM = 1000

ETAPAS = {
    'baixar': True, 'verificar': True, 'inspecionar': True, 'preprocessar': True, 'features': True,
    'fluxo': True, 'controles': True, 'p1': True, 'gcn': True,
    'redes': False,            # ShallowNet, EEGNet e EEGConformer: exigem GPU (ver pipeline_colab.ipynb)
    'comparar': True, 'auditoria': True, 'p2': True, 'pareado': True, 'sensibilidade': True,
    'variacoes': True, 'com_excluidas': True, 'externo': True, 'figuras': True,
}

registro = []


def rodar(*args, raiz=RAIZ):
    inicio = time.time()
    print(f'>>> {" ".join(map(str, args))}  ({time.strftime("%H:%M")})', flush=True)
    cli.main([*map(str, args), '--raiz', str(raiz), '--raiz-externa', str(RAIZ_EXTERNA)])
    registro.append({'etapa': ' '.join(map(str, args)), 'minutos': round((time.time() - inicio) / 60, 1)})

## 0. Ambiente

In [ ]:
faltando = []
for modulo in ['mne', 'mne_icalabel', 'mne_connectivity', 'openneuro', 'fooof', 'sklearn', 'statsmodels',
               'pyarrow', 'torch', 'braindecode', 'pyriemann', 'matplotlib']:
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltando.append(modulo)
print('python:', sys.executable, '| processos:', N_JOBS)
assert not faltando, f'instale as dependências que faltam: {faltando}'

## 1. Download e conferência dos arquivos

O `verificar` compara a duração de cada arquivo com a do JSON do BIDS e, nos BDF, o cabeçalho com o
tamanho real, porque o MNE lê arquivos truncados sem erro. Arquivos com problema são apagados e baixados
de novo. O centro externo é baixado numa raiz separada, para nunca entrar no treino.

In [ ]:
if ETAPAS['baixar']:
    rodar('baixar')
    rodar('baixar', '--datasets', 'ds007526', raiz=RAIZ_EXTERNA)
if ETAPAS['verificar']:
    from eeg_pd.dados import verificar
    conferencia = verificar(Caminhos(RAIZ))
    if conferencia['problema'].notna().any():
        verificar(Caminhos(RAIZ), apagar=True)
        rodar('baixar')
        conferencia = verificar(Caminhos(RAIZ))
    assert conferencia['problema'].isna().all(), 'ainda há arquivos com problema'
if ETAPAS['inspecionar']:
    rodar('inspecionar')

## 2. Pré-processamento

Cada gravação gera um log com o hash da configuração. Gravações já processadas com a configuração
atual são puladas; uma mudança em `config.py` refaz apenas as afetadas.

In [ ]:
if ETAPAS['preprocessar']:
    rodar('preprocessar', '--n-jobs', N_JOBS_PREPROCESSAMENTO)
    from eeg_pd.preprocessamento import carregar_logs
    display(carregar_logs(Caminhos(RAIZ)).groupby(['dataset', 'status']).size().rename('gravações'))

## 3. Features e fluxograma de participantes

In [ ]:
if ETAPAS['features']:
    rodar('features', '--n-jobs', N_JOBS)
if ETAPAS['fluxo']:
    rodar('fluxo')

## 4. Controles

Demografia, AUC usando só a idade, janelas sorteadas contra divisão por sujeito, classificador de
centro e teste de permutação do modelo de referência.

In [ ]:
if ETAPAS['controles']:
    rodar('controles', '--n-perm', N_PERM, '--n-jobs', N_JOBS)

## 5. P1: Parkinson vs. controle

In [ ]:
if ETAPAS['p1']:
    rodar('p1', '--med', 'ON', '--n-jobs', N_JOBS)
    rodar('p1', '--med', 'OFF', '--n-jobs', N_JOBS)
    display(pd.read_csv(RAIZ / 'resultados' / 'p1_ON' / 'resumo_por_centro.csv')
            .query("features == 'espectral' and modelo == 'logistica' and not harmonizacao")
            [['protocolo', 'centro', 'n', 'n_pd', 'auc', 'auc_ic', 'acuracia_balanceada']])
if ETAPAS['gcn']:
    rodar('profundos', '--modelo', 'gcn', 'gcn_sem_arestas')
if ETAPAS['redes']:
    rodar('profundos', '--modelo', 'shallownet', 'eegnet', 'conformer')

## 6. Comparação dos modelos e auditoria de vazamento

In [ ]:
if ETAPAS['comparar']:
    rodar('comparar')
if ETAPAS['auditoria']:
    rodar('auditoria')

## 7. P2: ON vs. OFF

Taxa de pares ordenados corretamente, com p por permutação (ON e OFF trocados dentro de pacientes
sorteados, refazendo toda a validação cruzada) e correção de Holm.

In [ ]:
if ETAPAS['p2']:
    rodar('p2', '--n-perm', N_PERM, '--n-jobs', N_JOBS)

## 8. Beta pareado em C3/C4: contraste, duração e deriva

In [ ]:
if ETAPAS['pareado']:
    rodar('pareado')
    rodar('blocos')

## 9. Sensibilidade e variações pré-especificadas

In [ ]:
if ETAPAS['sensibilidade']:
    rodar('sensibilidade', '--n-jobs', N_JOBS)
if ETAPAS['variacoes']:
    rodar('variacoes', '--n-jobs', N_JOBS)
if ETAPAS['com_excluidas']:   # inclui as sessões ON de pd6 e pd16
    rodar('p1', '--med', 'ON', '--com-excluidas', '--n-jobs', N_JOBS)
    rodar('comparar', '--com-excluidas')
    rodar('p2', '--com-excluidas', '--n-perm', N_PERM, '--n-jobs', N_JOBS)
    rodar('pareado', '--com-excluidas')
    rodar('blocos', '--com-excluidas')

## 10. Teste externo (Tel Aviv, ds007526)

O modelo principal é treinado uma vez com os três centros e aplicado uma vez ao centro externo.

In [ ]:
if ETAPAS['externo']:
    rodar('preprocessar', '--datasets', 'ds007526', '--n-jobs', N_JOBS_PREPROCESSAMENTO, raiz=RAIZ_EXTERNA)
    rodar('externo', '--n-jobs', N_JOBS)

## 11. Figuras

In [ ]:
if ETAPAS['figuras']:
    rodar('figuras')
    rodar('artigo')
display(pd.DataFrame(registro))